# DengueRadar — ETL epidemiológico e climático
Carregamos dados reais de TABNET/DATASUS e INMET, validamos períodos e preservamos registros ausentes. **Não substituímos ausências por zeros.** Verifique a abrangência geográfica e a convenção da semana epidemiológica do arquivo TABNET antes da publicação.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd().parent if Path.cwd().name.lower() in ('notebooks', 'notebook') else Path.cwd()
RAW = BASE / 'data' / 'raw'
PROCESSED = BASE / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
assert (RAW / 'datasus_dengue.csv').exists(), f"Arquivo ausente: {RAW / 'datasus_dengue.csv'}"
assert (RAW / 'inmet_clima.csv').exists(), f"Arquivo ausente: {RAW / 'inmet_clima.csv'}"
print('Diretório do projeto:', BASE)

## 1. Leitura e limpeza dos casos de dengue

In [ ]:
# A quantidade de linhas de rodapé varia por exportação TABNET: não usar skipfooter fixo.
df_dengue = pd.read_csv(RAW / 'datasus_dengue.csv', sep=';', encoding='latin1', engine='python', dtype=str)
df_dengue.columns = df_dengue.columns.str.strip()
coluna_semana = df_dengue.columns[0]
colunas_anos = [c for c in df_dengue.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4]
if not colunas_anos:
    raise ValueError('Não encontrei colunas de anos (ex.: 2023, 2024) no CSV DATASUS. Confira a exportação.')

df_casos = df_dengue.melt(id_vars=[coluna_semana], value_vars=colunas_anos, var_name='ano', value_name='casos_raw')
df_casos['semana_epidem'] = pd.to_numeric(df_casos[coluna_semana].astype(str).str.extract(r'(?i)semana\s*0*(\d{1,2})')[0], errors='coerce')
df_casos['ano'] = pd.to_numeric(df_casos['ano'], errors='coerce')
# Nota: '-' pode indicar zero, ausência ou dado não disponível, conforme o relatório de origem.
# Preservamos como NaN até confirmar essa codificação nos metadados.
valor = df_casos['casos_raw'].astype(str).str.strip().str.replace('.', '', regex=False)
valor = valor.str.replace(',', '.', regex=False)
df_casos['casos'] = pd.to_numeric(valor, errors='coerce')
df_casos = df_casos.dropna(subset=['ano', 'semana_epidem'])
df_casos = df_casos[df_casos['semana_epidem'].between(1, 53)]
df_casos['ano'] = df_casos['ano'].astype(int)
df_casos['semana_epidem'] = df_casos['semana_epidem'].astype(int)
df_casos = df_casos[['ano', 'semana_epidem', 'casos']]
if df_casos.duplicated(['ano', 'semana_epidem']).any():
    raise ValueError('Existem ano/semana duplicados no DATASUS. Verifique se a tabela contém mais de um território ou categoria.')
if (df_casos['casos'].dropna() < 0).any():
    raise ValueError('Foram encontrados casos negativos.')
print(f'Linhas válidas de período: {len(df_casos)} | casos ausentes: {df_casos.casos.isna().sum()}')
display(df_casos.head())

## 2. Clima INMET e correspondência de semanas
**Atenção:** semana ISO, semana epidemiológica e semana indicada pelo TABNET podem ter regras diferentes na virada do ano. A função abaixo usa semanas domingo–sábado, com a primeira semana ancorada no dia 1º de janeiro. A convenção precisa ser verificada contra a exportação do TABNET; diferenças nas viradas de ano devem ser reconciliadas antes de treinar.

In [ ]:
df_clima = pd.read_csv(RAW / 'inmet_clima.csv', sep=';', encoding='latin1', skiprows=8, decimal=',', low_memory=False)
df_clima.columns = df_clima.columns.str.strip()
col_data = next((c for c in df_clima.columns if c.lower().startswith('data')), None)
col_temp = next((c for c in df_clima.columns if 'TEMPERATURA DO AR - BULBO SECO' in c.upper()), None)
col_chuva = next((c for c in df_clima.columns if 'PRECIPITAÇÃO TOTAL' in c.upper()), None)
if not all([col_data, col_temp, col_chuva]):
    raise ValueError(f'Colunas INMET não reconhecidas: {df_clima.columns.tolist()}')
df_clima['Data'] = pd.to_datetime(df_clima[col_data], errors='coerce', dayfirst=False)
for origem, destino in [(col_temp, 'temp_media'), (col_chuva, 'precipitacao_total')]:
    df_clima[destino] = pd.to_numeric(df_clima[origem].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df_clima = df_clima.dropna(subset=['Data']).copy()

def semana_domingo(data):
    ano = data.year
    primeiro = pd.Timestamp(year=ano, month=1, day=1)
    inicio = primeiro - pd.Timedelta(days=(primeiro.weekday() + 1) % 7)
    return pd.Series({'ano': ano, 'semana_epidem': 1 + (data - inicio).days // 7})

semanas = df_clima['Data'].apply(semana_domingo)
df_clima[['ano','semana_epidem']] = semanas[['ano','semana_epidem']]
# ATENÇÃO: soma horária da chuva só é válida para série de uma estação e cobertura adequada.
# Dados horários de múltiplas estações exigem agregação espacial prévia.
clima_semanal = df_clima.groupby(['ano','semana_epidem'], as_index=False).agg(
    temp_media=('temp_media','mean'),
    precipitacao_total=('precipitacao_total', lambda s: s.sum(min_count=1)),
    medicoes_temp=('temp_media','count'),
    medicoes_chuva=('precipitacao_total','count')
)
# Padrão mínimo ilustrativo: pelo menos 70% das 168 leituras horárias esperadas por semana.
# Ajustar em caso de dados diários ou múltiplas estações e documentar essa escolha.
MIN_LEITURAS = 118
clima_semanal.loc[clima_semanal.medicoes_temp < MIN_LEITURAS, 'temp_media'] = np.nan
clima_semanal.loc[clima_semanal.medicoes_chuva < MIN_LEITURAS, 'precipitacao_total'] = np.nan
print('Semanas clima:', len(clima_semanal), '| Semanas com clima incompleto:', clima_semanal[['temp_media','precipitacao_total']].isna().any(axis=1).sum())
display(clima_semanal.head())

## 3. União, validação e exportação

In [ ]:
df_final = df_casos.merge(clima_semanal[['ano','semana_epidem','temp_media','precipitacao_total']], on=['ano','semana_epidem'], how='inner', validate='one_to_one')
antes = len(df_final)
df_final = df_final.dropna(subset=['casos','temp_media','precipitacao_total'])
df_final = df_final.sort_values(['ano','semana_epidem']).reset_index(drop=True)
if df_final.empty:
    raise ValueError('Base vazia após união e limpeza. Verifique anos, semanas e cobertura da estação meteorológica.')
if df_final.duplicated(['ano','semana_epidem']).any():
    raise ValueError('Ano/semana não é único.')
saida = PROCESSED / 'dataset_dengueradar_limpo.csv'
df_final.to_csv(saida, index=False)
print(f'Exportado: {saida} | {len(df_final)} semanas | {antes-len(df_final)} semanas removidas por dados ausentes')
print('Período:', (int(df_final.ano.min()), int(df_final.semana_epidem.min())), 'até', (int(df_final.ano.max()), int(df_final.semana_epidem.max())))
display(df_final.tail())